# A-RoF transmission

In [ ]:
import sys
from pathlib import Path

file_path    = str(Path.cwd())
project_path = str(Path.cwd().parent)

sys.path.append(project_path)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal             import welch, firwin, hilbert
from scipy.constants          import pi
from tqdm.notebook            import tqdm
from tabulate                 import tabulate

from optic.comm.ofdm          import modulateOFDM, demodulateOFDM
from optic.comm.modulation    import modulateGray
from optic.models.channels    import linearFiberChannel
from optic.models.devices     import mzm, photodiode
from optic.comm.metrics       import fastBERcalc, calcEVM
from optic.dsp.core           import pnorm, signal_power, finddelay, decimate, upsample, clockSamplingInterp
from optic.dsp.coreGPU        import firFilter
from optic.utils              import parameters, dBm2W

from dpd.channel_models       import power_amplifier
from dpd.calc_metrics         import calcACLR
from dpd.plots                import plotConst, plotSpec

In [ ]:
font = {'size':16}
plt.rc('font', **font)
plt.rcParams["font.family"] = "serif"

### 1 - OFDM signal generation

In [ ]:
# Parâmetros de modulação
modOrder  = 64                 # Modulation order
constType = 'qam'              # Constellation format
Rb = 5e9                       # Bitrate
SpS = 32

# OFDM parameters
paramOFDM = parameters()
paramOFDM.G   = 32              
paramOFDM.SpS = SpS

paramOFDM.Nfft = 2**10
paramOFDM.nullCarriers  = np.array([paramOFDM.Nfft//2], dtype = np.int64)
paramOFDM.pilotCarriers = np.linspace(0, paramOFDM.Nfft - 1, 32, dtype = np.int64)

Nz = len(paramOFDM.nullCarriers)
Np = len(paramOFDM.pilotCarriers)
Ni = paramOFDM.Nfft - Nz - Np
numOFDMframes = 100

In [ ]:
# Bits generation
np.random.seed(2)
bits = np.random.randint(2, size = (numOFDMframes*Ni, int(np.log2(modOrder))))

# Symbols modulation
symbTx = modulateGray(bits, modOrder, constType)
symbTx = pnorm(symbTx)

pilotSymb = 0.25*(max(symbTx.real) + 1j*max(symbTx.imag))
paramOFDM.pilot = pilotSymb

In [ ]:
# OFDM signal generation
sigTx = modulateOFDM(symbTx, paramOFDM)
sigTx = pnorm(sigTx)

Rs = Rb / ( Ni / (paramOFDM.Nfft + paramOFDM.G) * np.log2(modOrder))
Fs = Rs * paramOFDM.SpS

SpS_DPD = 4
Fs_DPD = Rs * SpS_DPD

t = np.arange(0, sigTx.size)*1/Fs
t_CP   = paramOFDM.SpS * paramOFDM.G * (1/Fs)
t_symb = paramOFDM.SpS * (paramOFDM.Nfft + paramOFDM.G) * (1/Fs)

In [ ]:
table = [['Property', 'Parameter', 'Value'],
         ['Sampling rate', 'Fs', f'{Fs*1e-9:.2f} Gsamples/s'],
         ['Symbol rate', 'Rs', f'{Rs*1e-9:.2f} Gsymb/s'],
         ['Bit rate', 'Rb', f'{Rb*1e-9:.2f} GHz'],
         ['Modulation order', 'M', f'{modOrder}'],
         ['Nfft', 'Nfft', f'{paramOFDM.Nfft}'],
         ['Cyclic prefix length', 'G', f'{paramOFDM.G}'],
         ['Number of information subcarriers', 'Ni', f'{Ni}'],
         ['Number of pilot subcarriers', 'Np', f'{Np}'],
         ['Number of null subcarriers', 'Nz', f'{Nz}'],
         ['Subcarriers spacing', 'Delta_f', f'{(Rs/paramOFDM.Nfft)*1e-3:.2f} kHz'],
         ['OFDM symbol period', 'T_symb', f'{1e6*t_symb:.2f} us'],
         ['Cyclic prefix duration', 'T_CP', f'{1e6*t_CP:.2f} us'] ]

print(tabulate(table, headers = 'firstrow', tablefmt = 'fancy_grid'))

In [ ]:
# MZM parameters
paramMZM = parameters()
paramMZM.Vpi = 3
paramMZM.Vb = -paramMZM.Vpi*0.5
paramMZM.P_laser = 0
paramMZM.Pin_MZM = 17

# RF parameters
paramRF = parameters()
paramRF.fc_e = 3.55e9
paramRF.bw = Rs
paramRF.Fs = Fs

# Optical fiber parameters
paramFiber = parameters()
paramFiber.L = 20
paramFiber.alpha = 0.2
paramFiber.D = 16
paramFiber.Fc = 193.1e12
paramFiber.Fs = Fs

# Photodiode parameters
paramPD = parameters()
paramPD.ideal = False
paramPD.B = paramRF.fc_e + 2*Rs
paramPD.Ipd_sat = 50e-3
paramPD.Fs = Fs

# PA parameters
paramPA = parameters()
paramPA.model_name = "modified_rapp"
paramPA.g = 16
paramPA.x_sat = 1.9
paramPA.sigma_p = 1.1
paramPA.alpha = -345
paramPA.beta = 0.17
paramPA.q = 4

filter_numtaps = 4096

### 2 - RoF channel

#### 2.1 RF signal

In [ ]:
sigTx_RF = np.real( sigTx * np.exp(1j * 2*pi * paramRF.fc_e * t) )
    
gain_pre_MZM = paramMZM.Pin_MZM - 10*np.log10(1e3*signal_power(sigTx_RF))

sigTx_RF = np.sqrt(10**((gain_pre_MZM)/10)) * sigTx_RF
sigTx_RF = np.clip(sigTx_RF, -paramMZM.Vpi/2, paramMZM.Vpi/2)

print(f"Power at MZM's input: {10*np.log10(1e3*signal_power(sigTx_RF)):.3f} dBm")

In [ ]:

indx = np.arange(0, 2000)

fig, axs = plt.subplots(1, 2, figsize = (15, 6))

axs[0].plot(t[indx]*1e9, sigTx_RF[indx], "k")
axs[1].psd(sigTx_RF, Fs = Fs/1e9, NFFT = 16*1024, sides = 'twosided', color = "k")

axs[0].set_xlim(0, t[indx][-1]*1e9)
axs[0].set_xlabel("t [ns]")
axs[0].set_ylabel("Tx RF")
axs[0].grid()
axs[0].minorticks_on()
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

axs[1].set_xlabel("f [GHz]")
axs[1].set_ylabel("Power Spectral Density [dB/Hz]")
axs[1].set_xlim(-paramRF.fc_e/1e9 - 2, paramRF.fc_e/1e9 + 2)
axs[1].grid(True)
axs[1].minorticks_on()
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

plt.tight_layout()
plt.show()

#### 2.2 Optical signal

In [ ]:
Ai     = np.sqrt(dBm2W(paramMZM.P_laser)) * np.ones(sigTx_RF.size)
sigTxo = mzm(Ai, sigTx_RF, paramMZM)

hopt_tx = firwin(filter_numtaps, paramRF.fc_e + 2*paramRF.bw, fs = Fs)
sigTxo  = np.sqrt(signal_power(sigTxo)) * pnorm(firFilter(hopt_tx, sigTxo))

sigRxo = linearFiberChannel(sigTxo, paramFiber)

In [ ]:
indx = np.arange(0, 2000)
fig, axs = plt.subplots(1, 2, figsize = (15, 6))

axs[0].plot(t[indx]*1e9, sigTxo[indx], "k")
axs[1].psd(np.abs(sigTxo), Fs = Fs/1e12, Fc = paramFiber.Fc/1e12, NFFT = 16*1024, sides = 'twosided', color = "k")

axs[0].set_xlim(0, t[indx][-1]*1e9)
axs[0].set_xlabel("t [ns]")
axs[0].set_ylabel("Optical Tx")
axs[0].grid()
axs[0].minorticks_on()
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

axs[1].set_xlabel("f [THz]")
axs[1].set_ylabel("Power Spectral Density [dB/Hz]")
axs[1].set_xlim(-(paramRF.fc_e + 2.5*paramRF.bw - paramFiber.Fc)/1e12, (paramRF.fc_e + 2.5*paramRF.bw + paramFiber.Fc)/1e12)
axs[1].set_ylim(-100)
axs[1].grid(True)
axs[1].minorticks_on()
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

plt.tight_layout()
plt.show()

#### 2.3 Electrical signal (photocurrent)

In [ ]:
I_Rx = photodiode(sigRxo, paramPD)
I_Rx -= I_Rx.mean()

In [ ]:

indx = np.arange(0, 2000)
fig, axs = plt.subplots(1, 2, figsize = (15, 6))

axs[0].plot(t[indx]*1e9, I_Rx[indx], "k")
axs[1].psd(I_Rx, Fs = Fs/1e9, NFFT = 16*1024, sides = 'twosided', color = "k")

axs[0].set_xlim(0, t[indx][-1]*1e9)
axs[0].set_xlabel("t [ns]")
axs[0].set_ylabel("I-Rx")
axs[0].grid()
axs[0].minorticks_on()
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

axs[1].set_xlabel("f [GHz]")
axs[1].set_ylabel("Power Spectral Density [dB/Hz]")
axs[1].set_xlim(- paramRF.fc_e/1e9 - 2,  paramRF.fc_e/1e9 + 2)
axs[1].set_ylim(-150)
axs[1].grid(True)
axs[1].set_yticks(np.arange(-150, -80, 10))
axs[1].minorticks_on()
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

plt.tight_layout()
plt.show()

#### 2.4 Photocurrent after BPF

In [ ]:
# Bandpass filter
hbp_RF = firwin(filter_numtaps, (paramRF.fc_e - 2*paramRF.bw, paramRF.fc_e + 2*paramRF.bw), pass_zero = 'bandpass', fs = Fs)

I_RF = firFilter(hbp_RF, I_Rx)

In [ ]:

indx = np.arange(0, 2000)
fig, axs = plt.subplots(1, 2, figsize = (15, 6))

axs[0].plot(t[indx]*1e9, I_RF[indx], "k")
axs[1].psd(I_RF, Fs = Fs/1e9, NFFT = 16*1024, sides = 'twosided', color = "k")

axs[0].set_xlim(0, t[indx][-1]*1e9)
axs[0].set_xlabel("t [ns]")
axs[0].set_ylabel("I-RF")
axs[0].grid()
axs[0].minorticks_on()
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

axs[1].set_xlabel("f [GHz]")
axs[1].set_ylabel("Power Spectral Density [dB/Hz]")
axs[1].set_xlim(- paramRF.fc_e/1e9 - 2,  paramRF.fc_e/1e9 + 2)
axs[1].set_ylim(-150)
axs[1].grid(True)
axs[1].set_yticks(np.arange(-150, -80, 10))
axs[1].minorticks_on()
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

plt.tight_layout()
plt.show()

#### 2.5 Electrical signal after-PA

In [ ]:
sigRx = hilbert(I_RF)*np.exp(-1j * 2*pi * paramRF.fc_e * t) * 1e3
    
# Power amplification    
sigRx = power_amplifier(sigRx, paramPA)

sigRx_PA = sigRx.copy()
sigRx_PA = pnorm(sigRx_PA)

hlp = firwin(filter_numtaps, Rs/1.75, fs = Fs)
sigRx = firFilter(hlp, sigRx)

In [ ]:

indx = np.arange(0, 2000)
fig, axs = plt.subplots(1, 2, figsize = (15, 6))

axs[0].plot(t[indx]*1e9, np.abs(sigRx_PA[indx]), "k", lw = 2)
axs[1].psd(sigRx_PA, Fs = Fs/1e9, NFFT = 16*1024, sides = 'twosided', color = "k")

axs[0].set_xlim(0, t[indx][-1]*1e9)
axs[0].set_xlabel("t [ns]")
axs[0].set_ylabel("Rx after-PA")
axs[0].set_ylim(0)
axs[0].grid()
axs[0].minorticks_on()
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[0].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

axs[1].set_xlabel("f [GHz]")
axs[1].set_ylabel("Power Spectral Density [dB/Hz]")
axs[1].set_xlim(-2, 2)
axs[1].set_ylim(-38, 10)
axs[1].grid(True)
axs[1].set_yticks(np.arange(-35, 10, 10))
axs[1].minorticks_on()
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")

plt.tight_layout()
plt.show()

In [ ]:
delay = finddelay(sigRx, sigTx)
sigRx = np.roll(sigRx, -delay)

rot = np.mean(sigTx/sigRx)
sigRx = rot/np.abs(rot)*sigRx

In [ ]:

indx = np.arange(0, 2000)
fig, axs = plt.subplots(1, 2, figsize = (15, 6))

axs[0].plot(t[indx]*1e9, np.abs(pnorm(sigTx))[indx], label = "Tx", lw = 2, color = "k")
axs[0].plot(t[indx]*1e9, np.abs(pnorm(sigRx))[indx], label = "Rx", lw = 2, color = "b")

axs[1].psd(pnorm(sigTx), Fs = Fs/1e9, NFFT = 16*1024, sides = 'twosided', label = "Tx", color = "k")
axs[1].psd(pnorm(sigRx), Fs = Fs/1e9, NFFT = 16*1024, sides = 'twosided', label = "Rx", color = "b")

axs[0].set_xlim(0, t[indx][-1]*1e9)
axs[0].set_xlabel("t [ns]")
axs[0].set_ylim(0)
axs[0].grid()
axs[0].legend(framealpha = 1)

axs[1].set_xlabel("f [GHz]")
axs[1].set_ylabel("Power Spectral Density [dB/Hz]")
axs[1].set_xlim(-2, 2)
axs[1].set_ylim(-38, 10)
axs[1].grid(True)
axs[1].set_yticks(np.arange(-35, 10, 10))
axs[1].minorticks_on()
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='minor',  width=1, direction = "in")
axs[1].tick_params(axis = 'both', top = "True", right = "True", which='major',  width=1.5, direction = "in")
axs[1].legend(framealpha = 1)

plt.tight_layout()
plt.show()

#### 2.6 Performance evaluation

In [ ]:
# Parâmetros da decimação
paramDec = parameters()
paramDec.SpS_in  = paramOFDM.SpS
paramDec.SpS_out = 1

symbRx_OFDM = decimate(sigRx, paramDec).ravel()
symbRx = demodulateOFDM(symbRx_OFDM, paramOFDM)

In [ ]:
discard = 100
index = np.arange(discard, symbTx.size - discard)

BER, _, _ = fastBERcalc(symbRx[index], symbTx[index], modOrder, constType)
EVM = np.sqrt(calcEVM(symbRx, modOrder, constType))*100

print(f"BER = {BER[0]:.5f} \nEVM = {EVM[0]:.3f} %")

# Plot normalization
symbTx = pnorm(symbTx)
symbRx = pnorm(symbRx)

plotConst(symbTx, symbRx, axs_lim = 1.5)

In [ ]:
freq, P_sigTx = welch(pnorm(clockSamplingInterp(sigTx.reshape(-1, 1), Fs, Fs/(SpS/4)).ravel()),\
                      fs = Fs/(SpS/4), nfft = 16*1024, return_onesided = False)

freq, P_sigRx_PA = welch(pnorm(clockSamplingInterp(sigRx_PA.reshape(-1, 1), Fs, Fs/(SpS/4)).ravel()),\
                         fs = Fs/(SpS/4), nfft = 16*1024, return_onesided = False)

print(f"SigTx - ACLR = {calcACLR(P_sigTx, freq, Rs/2, 10e6):.3f} dB")
print(f"SigRx PA w/o DPD - ACLR = {calcACLR(P_sigRx_PA, freq, Rs/2, 10e6):.3f} dB")

plotSpec(freq, P_sigTx, P_sigRx_PA, label = "No DPD")